#Data Aggregation

The last stage of data manipulation is data aggregation. For data aggregation you generally mean a transformation that produces a single integer from an array. In fact, you have already made many operations
of data aggregation, for example, when we calculated the sum(), mean(), count(). In fact, these functions operate on a set of data and shall perform a calculation with a consistent result consisting of a single value. However, a more formal manner and the one with more control in data aggregation is that which includes
the categorization of a set.

The categorization of a set of data carried out for grouping is often a critical stage in the process of data analysis. It is a process of transformation since after the division into different groups, you apply a function that converts or transforms the data in some way depending on the group they belong to. Very often the two phases of grouping and application of a function are performed in a single step.

Also for this part of the data analysis, pandas provides a tool very flexible and high performance:
GroupBy.

#GroupBy

Now you will analyze in detail what the process of GroupBy is and how it works. Generally, it refers to its internal mechanism as a process called SPLIT-APPLY-COMBINE. So in its pattern of operation you may conceive this process as divided into three different phases expressed precisely by three operations:

• splitting: division into groups of datasets

• applying: application of a function on each group

• combining: combination of all the results obtained by different groups

Analyze better the three different phases . In the first phase, that of splitting, the data contained within a data structure, such as a Series or a DataFrame, are divided into several groups,
according to a given criterion, which is often linked to indexes or just certain values in a column. In the jargon of SQL, values contained in this column are reported as keys. Furthermore, if you are working with two-dimensional objects such as the DataFrame, the grouping criterion may be applied both to the line (axis = 0) for that column (axis = 1).

The second phase, that of applying, consists in applying a function, or better a calculation expressed precisely by a function, which will produce a new and single value, specific to that group.

The last phase, that of combining, will collect all the results obtained from each group and combine them together to form a new object.

#A Practical Example

You have just seen that the process of data aggregation in pandas is divided into various phases calls precisely split-apply-combine. With these pandas are not expressed explicitly with the functions as you would have expected, but by a groupby() function that generates an GroupBy object then that is the core of the whole process.

But to understand this mechanism, you must switch to a practical example. So, first, define a DataFrame containing both numeric and string values.

In [ ]:
frame = pd.DataFrame({ 'color': ['white','red','green','red','green'],
... 'object': ['pen','pencil','pencil','ashtray','pen'],
... 'price1' : [5.56,4.20,1.30,0.56,2.75],
... 'price2' : [4.75,4.12,1.60,0.75,3.15]})
>>> frame

,color,object,price1,price2
0,white,pen,5.56,4.75
1,red,pencil,4.20,4.12
2,green,pencil,1.30,1.60
3,red,ashtray,0.56,0.75
4,green,pen,2.75,3.15


Suppose you want to calculate the average price1 column using group labels listed in the column color. There are several ways to do this. You can for example access the price1 column and call the groupby() function with the column color.

In [ ]:
group = frame['price1'].groupby(frame['color'])
group

The object that we got is a GroupBy object. In the operation that you just did there was not really any calculation; there was just a collection of all the information needed to calculate to be executed. What you
have done is in fact a process of grouping, in which all rows having the same value of color are grouped into a single item.

To analyze in detail how the division into groups of rows of DataFrame was made, you call the attribute groups GroupBy object.

In [ ]:
group.groups

{'green': [2, 4], 'red': [1, 3], 'white': [0]}

As you can see, each group is listed explicitly specifying the rows of the data frame assigned to each of them. Now it is sufficient to apply the operation on the group to obtain the results for each individual group.

In [ ]:
group.mean()

,price1
color,
green,2.025
red,2.380
white,5.560


In [ ]:
group.sum()

,price1
color,
green,4.05
red,4.76
white,5.56


#Hierarchical Grouping

You have seen how to group the data according to the values of a column as a key choice. The same thing can be extended to multiple columns, i.e., make a grouping of multiple keys hierarchical.

In [ ]:
ggroup = frame['price1'].groupby([frame['color'],frame['object']])
ggroup.groups

{('green', 'pen'): [4], ('green', 'pencil'): [2], ('red', 'ashtray'): [3], ('red', 'pencil'): [1], ('white', 'pen'): [0]}

In [ ]:
ggroup.sum()

color  object 
green  pen        2.75
       pencil     1.30
red    ashtray    0.56
       pencil     4.20
white  pen        5.56
Name: price1, dtype: float64

So far you have applied the grouping to a single column of data, but in reality it can be extended to multiple columns or the entire data frame. Also if you do not need to reuse the object GroupBy several times, it is convenient to combine in a single passing all of the grouping and calculation to be done, without defining any intermediate variable.

In [ ]:
frame[['price1','price2']].groupby(frame['color']).mean()

,price1,price2
color,,
green,2.025,2.375
red,2.380,2.435
white,5.560,4.750


#Group Iteration

The GroupBy object supports the operation of an iteration for generating a sequence of 2-tuples containing the name of the group together with the data portion.

In [ ]:
for name, group in frame.groupby('color'):
    print(name)
    print(group)

green
   color  object  price1  price2
2  green  pencil    1.30    1.60
4  green     pen    2.75    3.15
red
  color   object  price1  price2
1   red   pencil    4.20    4.12
3   red  ashtray    0.56    0.75
white
   color object  price1  price2
0  white    pen    5.56    4.75


#Functions on Groups

Although many methods have not been implemented specifically for use with GroupBy, they actually work correctly with data structures as the Series. You saw in the previous section how easy it is to get the Series by a GroupBy object, specifying the name of the column and then by applying the method to make the calculation. For example, you can use the calculation of quantiles with the quantiles() function.

In [ ]:
group = frame.groupby('color')
group['price1'].quantile(0.6)

,price1
color,
green,2.170
red,2.744
white,5.560


You can also define their own aggregation functions. Define the function separately and then you pass as an argument to the mark() function. For example, you could calculate the range of the values of each group.

In [ ]:
def range(series):
  return series.max() - series.min()

group['price1'].agg(range)

,price1
color,
green,1.45
red,3.64
white,0.00


1. Defining the Custom FunctionPythondef range(series):

  return series.max() - series.min()

The Goal: Standard pandas doesn't have a built-in .range() method like it does for .mean() or .sum()

The Logic: This function takes a "series" (a column of numbers) and calculates the spread using the formula:$$Range = \text{max}(x) - \text{min}(x)$$A Small

Note: In Python, range is a built-in keyword (used for loops). While this code works, it "shadows" the built-in function. In a professional setting, naming it something like get_range or ptp (peak-to-peak) is usually better to avoid confusion.

2. Applying the AggregationPythongroup['price1'].agg(range)
This line assumes you have already grouped your data (e.g., group = df.groupby('category'))

.group['price1']: This selects the specific column named 'price1' from your grouped object.

.agg(range): The .agg() (aggregate) method tells pandas to run a specific function on every group separately.

How it works behind the scenes:

Split: The data is split into chunks based on a grouping key (like "Store A", "Store B")

Apply: Your custom range function is applied to the 'price1' column for each of those chunks.

Combine: The results are combined into a new Series or DataFrame where the index is your group and the value is the calculated range.

The agg() function() allows you to use aggregate functions on an entire DataFrame.

In [ ]:
def my_range(series):
  return series.max() - series.min()

group = frame.groupby('color')
group[['price1', 'price2']].agg(my_range)

,price1,price2
color,,
green,1.45,1.55
red,3.64,3.37
white,0.00,0.00


Also you can use more aggregate functions at the same time always with the mark() function passing an array containing the list of operations to be done, which will become the new columns.

In [ ]:
group['price1'].agg(['mean','std',range])

,mean,std,range
color,,,
green,2.025,1.025305,1.45
red,2.380,2.573869,3.64
white,5.560,NaN,0.00


Advanced Data Aggregation
In this section you will be introduced to transform() and apply() functions, which will allow you to perform many kinds of group operations, some very complex.

Now suppose we want to bring together in the same DataFrame the following: (i) the DataFrame of
origin (the one containing the data) and (ii) that obtained by the calculation of group aggregation, for
example, the sum.

In [ ]:
frame = pd.DataFrame({ 'color':['white','red','green','red','green'],
... 'price1':[5.56,4.20,1.30,0.56,2.75],
... 'price2':[4.75,4.12,1.60,0.75,3.15]})
frame

,color,price1,price2
0,white,5.56,4.75
1,red,4.20,4.12
2,green,1.30,1.60
3,red,0.56,0.75
4,green,2.75,3.15


In [ ]:
sums = frame.groupby('color').sum().add_prefix('tot_')
sums

,tot_price1,tot_price2
color,,
green,4.05,4.75
red,4.76,4.87
white,5.56,4.75


In [ ]:
pd.merge(frame,sums,left_on='color',right_index=True)

,color,price1,price2,tot_price1,tot_price2
0,white,5.56,4.75,5.56,4.75
1,red,4.20,4.12,4.76,4.87
2,green,1.30,1.60,4.05,4.75
3,red,0.56,0.75,4.76,4.87
4,green,2.75,3.15,4.05,4.75


Gemini said
This line of code is used to combine (join) two different datasets based on a shared key. It is specifically designed to align a regular column in one table with the index of another table.

In data analysis, this is frequently done to "broadcast" aggregated results (like the sums or ranges you calculated earlier) back onto the original raw data.

Line-by-Line Breakdown
pd.merge(...)
This is the main pandas function used to join two DataFrames together, similar to a JOIN operation in SQL.

frame
This is the Left DataFrame. Usually, this is your original "long" dataset that contains many rows, including a column named 'color'.

sums
This is the Right DataFrame. Based on your previous code, this is likely the result of a groupby operation. In that case, the colors aren't in a column—they are the index (the row labels).

left_on='color'
This tells pandas: "Look at the column named 'color' in the left table (frame) to find the values we want to match."

right_index=True
This tells pandas: "Instead of looking for a column in the right table (sums), use its index to find the matching values."

So thanks to the merge(), you managed to add the results of a calculation of aggregation in each line of the data frame to start. But actually there is another way to do this type of operation. That is by using the transform(). This function performs the calculation of aggregation as you have seen before, but at the same
time shows the values calculated based on the key value on each line of the data frame to start.

In [ ]:
frame.groupby('color').transform(np.sum).add_prefix('tot_')

/tmp/ipykernel_4822/147351506.py:1: FutureWarning: The provided callable <function sum at 0x7c979d528400> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  frame.groupby('color').transform(np.sum).add_prefix('tot_')


,tot_price1,tot_price2
0,5.56,4.75
1,4.76,4.87
2,4.05,4.75
3,4.76,4.87
4,4.05,4.75


As you can see the transform() method is a more specialized function that has very specific requirements: the function passed as an argument must produce a single scalar value (aggregation) to be broadcasted.

The method to cover more general GroupBy is applicable to apply(). This method applies in its entirety the scheme split-apply-combine. In fact, this function divides the object into parts in order
to be manipulated, invokes the passage of function on each piece, and then tries to chain together the various parts.

In [ ]:
frame = pd.DataFrame( { 'color':['white','black','white','white','black','black'],
... 'status':['up','up','down','down','down','up'],
... 'value1':[12.33,14.55,22.34,27.84,23.40,18.33],
... 'value2':[11.23,31.80,29.99,31.18,18.25,22.44]})
frame

,color,status,value1,value2
0,white,up,12.33,11.23
1,black,up,14.55,31.80
2,white,down,22.34,29.99
3,white,down,27.84,31.18
4,black,down,23.40,18.25
5,black,up,18.33,22.44


In [ ]:
frame.groupby(['color','status']).apply( lambda x: x.max())

/tmp/ipykernel_4822/1415347647.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frame.groupby(['color','status']).apply( lambda x: x.max())


color status  value1  value2
color status                              
black down    black   down   23.40   18.25
      up      black     up   18.33   31.80
white down    white   down   27.84   31.18
      up      white     up   12.33   11.23

In [ ]:
reindex = {
    0: 'first',
    1: 'second',
    2: 'third',
    3: 'fourth',
    4: 'fifth'
}
recolumn = {'item':'object','price': 'value'}
frame.rename(index=reindex, columns=recolumn)

,color,status,value1,value2
first,white,up,12.33,11.23
second,black,up,14.55,31.80
third,white,down,22.34,29.99
fourth,white,down,27.84,31.18
fifth,black,down,23.40,18.25
5,black,up,18.33,22.44


In [ ]:
temp = pd.date_range('1/1/2015', periods=10, freq= 'h')
temp

DatetimeIndex(['2015-01-01 00:00:00', '2015-01-01 01:00:00',
               '2015-01-01 02:00:00', '2015-01-01 03:00:00',
               '2015-01-01 04:00:00', '2015-01-01 05:00:00',
               '2015-01-01 06:00:00', '2015-01-01 07:00:00',
               '2015-01-01 08:00:00', '2015-01-01 09:00:00'],
              dtype='datetime64[ns]', freq='h')

The Breakdown
temp = pd.date_range('1/1/2015', periods=10, freq='h')

pd.date_range(): This is the pandas constructor for creating a range of dates. It works similarly to Python's built-in range() function but specifically for time.

'1/1/2015': This is the start date. Pandas is very flexible with date strings; you could also write "January 1st, 2015" or "2015-01-01".

periods=10: This tells pandas exactly how many timestamps to generate.

freq='h': This sets the frequency. Here, 'h' stands for hourly.

In [ ]:
timetable = pd.DataFrame( {'date': temp, 'value1' : np.random.rand(10),'value2' : np.random.rand(10)})
timetable

,date,value1,value2
0,2015-01-01 00:00:00,0.403374,0.450414
1,2015-01-01 01:00:00,0.516152,0.699781
2,2015-01-01 02:00:00,0.021874,0.776259
3,2015-01-01 03:00:00,0.775214,0.568929
4,2015-01-01 04:00:00,0.765914,0.143994
5,2015-01-01 05:00:00,0.348902,0.724520
6,2015-01-01 06:00:00,0.540106,0.972983
7,2015-01-01 07:00:00,0.458453,0.165245
8,2015-01-01 08:00:00,0.873727,0.562233
9,2015-01-01 09:00:00,0.934986,0.614950


In [ ]:
timetable['cat'] = ['up','down','left','left','up','up','down','right','right','up']
timetable

,date,value1,value2,cat
0,2015-01-01 00:00:00,0.403374,0.450414,up
1,2015-01-01 01:00:00,0.516152,0.699781,down
2,2015-01-01 02:00:00,0.021874,0.776259,left
3,2015-01-01 03:00:00,0.775214,0.568929,left
4,2015-01-01 04:00:00,0.765914,0.143994,up
5,2015-01-01 05:00:00,0.348902,0.724520,up
6,2015-01-01 06:00:00,0.540106,0.972983,down
7,2015-01-01 07:00:00,0.458453,0.165245,right
8,2015-01-01 08:00:00,0.873727,0.562233,right
9,2015-01-01 09:00:00,0.934986,0.614950,up
